# PARTE I: INFORME DEL PROYECTO---## 1. Portada**SentinelGrafo — Clasificación de mensajes críticos en plataformas digitales mediante Redes Neuronales Gráficas**- **Curso:** Inteligencia Artificial 2026-1, Sección 3, EPISW-FISI-UNMSM- **Profesor:** Rolando A. Maguiña Pérez- **Integrantes (con código):**  - Segovia Valencia Jim Bryan  - Morales Usca Andres  - Chavez Cerna Joshua---

## 2. Resumen*(200-300 palabras — síntesis del proyecto: qué se hizo, con qué técnicas, sobre qué datasets, principales resultados.)*> **Pendiente:** Redactar el resumen una vez completados los experimentos y el análisis de resultados.

## 3. Objetivo### Objetivo generalAplicar Redes Neuronales Gráficas (GNNs) a la clasificación de texto en dos casos de estudio, comparando su desempeño con un baseline clásico (Regresión Logística sobre TF-IDF).### Objetivos específicos1. Implementar un pipeline de construcción de grafos textuales mediante TF-IDF y k-NN con similitud coseno.2. Entrenar y evaluar modelos GraphSAGE y GCN sobre los grafos construidos para cada caso de estudio.3. Comparar el desempeño de las GNNs frente al baseline mediante métricas de clasificación (accuracy, precision, recall, F1-score).4. Desarrollar una interfaz interactiva (dashboard) para predicción en vivo y exploración visual de los grafos.

## 4. Marco Teórico### 4.1 Redes Neuronales Gráficas (GNNs)*(Definición, paradigma de Message Passing: AGGREGATE + UPDATE, aplicaciones.)*> **Pendiente:** Completar con la redacción de tu compañero.### 4.2 GraphSAGE (Graph Sample and AggregatE)*(Fundamentos, muestreo de vecinos, función de agregación, carácter inductivo.)*> **Pendiente:** Completar con la redacción de tu compañero.### 4.3 GCN (Graph Convolutional Networks)*(Convolución espectral de primer orden, diferencias con GraphSAGE.)*> **Pendiente:** Completar con la redacción de tu compañero.### 4.4 Construcción de grafos textuales*(TF-IDF, similitud coseno, k-NN para conectar documentos como nodos de un grafo.)*> **Pendiente:** Completar con la redacción de tu compañero.### 4.5 Comparativa con técnicas similares*(GAT — mención teórica, por qué se eligió GraphSAGE como modelo principal.)*> **Pendiente:** Completar con la redacción de tu compañero.

## 5. Desarrollo y Aplicación### 5.1 Pipeline generalEl flujo de trabajo de SentinelGrafo transforma documentos de texto en un grafo y aplica GNNs para clasificarlos. El pipeline consta de cinco etapas:```Documento → Limpieza → Vectorización TF-IDF → Grafo k-NN → GNN → Clase predicha```#### Etapa 1 — Limpieza de textoCada texto se normaliza mediante la función `clean_text()`:- Conversión a minúsculas.- Eliminación de URLs (`https?://...`) y etiquetas HTML (`<...>`).- Eliminación de puntuación y caracteres no alfabéticos (conservando letras con tilde y ñ).- Colapso de espacios múltiples en uno solo.Esta limpieza es idéntica para ambos casos de estudio y se aplica antes de cualquier vectorización.#### Etapa 2 — Vectorización TF-IDFSe utiliza `TfidfVectorizer` de scikit-learn con *stop words* en inglés. Cada documento se representa como un vector numérico de dimensión fija:- **Caso 1 (Disaster Tweets):** 3,000 características.- **Caso 2 (AG News):** 5,000 características (vocabulario más amplio por la mayor longitud de los textos).La matriz TF-IDF resultante tiene dimensiones `[n_documentos × n_features]`. Estos vectores son las *features* iniciales de cada nodo en el grafo.#### Etapa 3 — Construcción del grafo k-NNLa función `build_graph()` convierte el conjunto de documentos en un objeto `torch_geometric.data.Data`:1. Se calcula la matriz de similitud coseno entre todos los pares de documentos usando sus vectores TF-IDF.2. Con `NearestNeighbors` de scikit-learn (`metric='cosine'`), se encuentran los k vecinos más cercanos de cada documento.3. Cada documento se convierte en un **nodo** del grafo.4. Se crean **aristas dirigidas** desde cada nodo hacia sus k vecinos más similares (se excluye el auto-bucle).5. El grafo resultante se empaqueta en `Data(x=features_TFIDF, edge_index=aristas, y=etiquetas)`.Para ambos casos se usó **k = 5**. Esto produce grafos conexos sin densidad excesiva:- Caso 1: 6,542 nodos y 32,710 aristas (promedio de 5 aristas salientes por nodo).- Caso 2: 24,000 nodos y 120,000 aristas.#### Etapa 4 — Modelos GNNSe implementó una clase unificada `GNNClassifier` que soporta dos arquitecturas intercambiables mediante el parámetro `model_type`:| Arquitectura | Capa PyG | Característica principal ||-------------|----------|--------------------------|| **GraphSAGE** | `SAGEConv` | Inductivo: aprende una función de agregación sobre vecinos muestreados. Puede generalizar a nodos no vistos durante el entrenamiento. || **GCN** | `GCNConv` | Transductivo: aplica convolución espectral de primer orden sobre todo el grafo. Requiere el grafo completo para inferencia. |Ambos modelos comparten **exactamente la misma estructura**:- **Capa 1:** Convolución de grafo (SAGEConv o GCNConv) con entrada = dimensiones TF-IDF, salida = 64.- **ReLU + Dropout 0.5**.- **Capa 2:** Convolución de grafo con entrada = 64, salida = 64.- **ReLU + Dropout 0.5**.- **Clasificador:** Capa lineal `(64 → n_clases)`.El método `get_embeddings()` extrae las representaciones latentes de 64 dimensiones de la penúltima capa para visualización t-SNE.#### Etapa 5 — EntrenamientoLa función `train_and_evaluate()` gestiona el ciclo completo de entrenamiento para un modelo dado:- **Optimizador:** Adam con learning rate = 0.01 y weight decay = 5×10⁻⁴ (regularización L2).- **Función de pérdida:** Cross-Entropy Loss.- **Early stopping:** paciencia de 30 épocas sobre accuracy de validación. Si no hay mejora en 30 épocas consecutivas, se restaura el mejor modelo y se detiene el entrenamiento.- **Máximo de épocas:** 200 (rara vez se alcanza por el early stopping).- **Split:** 80% entrenamiento / 20% prueba, estratificado por clase (`random_state=42`).- **Máscaras:** `train_mask` y `test_mask` binarias sobre los índices del grafo.Se entrenan **cuatro modelos en total**: GraphSAGE y GCN para cada uno de los dos casos de estudio.#### Etapa 6 — Inferencia sobre textos nuevosPara clasificar un texto no visto durante el entrenamiento, se implementa inferencia por **subgrafo local**:1. Se limpia el nuevo texto con `clean_text()`.2. Se vectoriza con el mismo `TfidfVectorizer` usado en entrenamiento.3. Se calcula la similitud coseno contra todos los documentos del dataset y se seleccionan los k vecinos más cercanos.4. El nuevo nodo se conecta al grafo existente a través de esos k vecinos ancla.5. Con `k_hop_subgraph` de PyTorch Geometric se extrae el subgrafo de 2 saltos alrededor del nuevo nodo (porque la GNN tiene 2 capas).6. Se ejecuta el forward pass de la GNN solo sobre ese subgrafo local, obteniendo la clase predicha y la confianza.Este enfoque es **teóricamente correcto** porque en una GNN de L capas, la predicción de un nodo depende exclusivamente de su vecindario de L saltos. No es necesario re-ejecutar el modelo sobre el grafo completo, lo que lo hace eficiente para uso en producción.#### Interfaz de usuario — Dashboard interactivoSe desarrolló un panel interactivo con `ipywidgets` que contiene dos pestañas:**Pestaña 1 — Predicción en Vivo:**- Selector de caso de estudio (Disaster Tweets o AG News).- Campo de texto para ingresar un tweet o noticia en inglés.- Botón PREDECIR: ejecuta la inferencia por subgrafo local.- Muestra: clase predicha, nivel de confianza, barras de probabilidad por clase, y los vecinos ancla más similares del dataset.- Visualización del subgrafo local usado en la predicción (nodo nuevo en estrella, vecinos ancla resaltados).**Pestaña 2 — Explorador de Grafos:**- Permite visualizar interactivamente los grafos k-NN de ambos casos.- Controles: número de nodos, k vecinos, tamaño de nodo, opacidad de aristas, capas BFS visibles.- Modos de coloración: por clases reales o por comunidades detectadas (Louvain/greedy modularity).- Opción de destacar una clase específica (atenúa las demás).- Resumen estadístico del subgrafo visualizado (nodos, aristas, grado promedio).---### 5.2 Caso de estudio 1 — Disaster Tweets**Problema:** Clasificar tweets como desastre real (`target=1`) o uso figurado/irrelevante (`target=0`). Ejemplo:- *"Earthquake struck Japan this morning"* → **Desastre real**.- *"My day was a total disaster lol"* → **No desastre** (uso figurado).**Origen del dataset:** Desafío *Natural Language Processing with Disaster Tweets* de Kaggle. Es un benchmark estándar para clasificación binaria de texto corto en el dominio de respuesta a emergencias.**Preprocesamiento aplicado:**- Carga del archivo `train.csv` (7,613 tweets, 5 columnas: `id`, `keyword`, `location`, `text`, `target`).- Distribución original: 4,342 no-desastre (57%), 3,271 desastre real (43%).- **Balanceo:** se igualan ambas clases a 3,271 muestras cada una (6,542 total) mediante muestreo aleatorio de la clase mayoritaria.- Limpieza de texto con `clean_text()`. Se eliminan tweets vacíos tras limpieza.- Vectorización TF-IDF con `max_features=3000`.- Construcción de grafo k-NN con `k=5` vecinos y métrica coseno.- Split estratificado 80/20: 5,233 train / 1,309 test.**Modelos:** GraphSAGE + GCN, 2 capas, 64 dimensiones ocultas, dropout 0.5.**Baseline:** Regresión Logística sobre los vectores TF-IDF (sin información del grafo).**Relevancia:** Este caso es representativo de aplicaciones reales de monitoreo de redes sociales para alerta temprana de desastres, donde distinguir reportes genuinos del uso metafórico del lenguaje es crítico para evitar falsas alarmas.---### 5.3 Caso de estudio 2 — AG News**Problema:** Clasificar noticias en 4 categorías temáticas:- **World (Mundo):** noticias internacionales y política.- **Sports (Deportes):** noticias deportivas.- **Business (Negocios):** noticias económicas y financieras.- **Sci/Tech (Ciencia/Tecnología):** noticias de ciencia y tecnología.**Origen del dataset:** AG News corpus, creado por Zhang et al. (2015). Contiene titulares y descripciones de noticias recolectadas de Internet. Es un benchmark estándar para clasificación de texto multiclase.**Preprocesamiento aplicado:**- Carga del archivo `ag_news_train.csv` (120,000 noticias, 3 columnas: `label`, `title`, `description`).- Las etiquetas originales 1-4 se convierten a 0-3.- Distribución original perfectamente balanceada: 30,000 noticias por clase.- **Muestreo:** para mantener baja complejidad computacional se seleccionan 6,000 noticias por clase (24,000 total).- Concatenación de título y descripción en un solo campo `content`.- Limpieza de texto con `clean_text()`. Se eliminan documentos vacíos.- Vectorización TF-IDF con `max_features=5000` (mayor dimensionalidad que el Caso 1 por la mayor longitud y diversidad léxica de las noticias).- Construcción de grafo k-NN con `k=5` vecinos y métrica coseno.- Split estratificado 80/20: 19,200 train / 4,800 test.**Modelos:** GraphSAGE + GCN, 2 capas, 64 dimensiones ocultas, dropout 0.5.**Baseline:** Regresión Logística sobre los vectores TF-IDF (sin información del grafo).**Relevancia:** Este caso demuestra la aplicabilidad de GNNs a un problema multiclase con textos de mayor longitud y estructura formal (noticias vs tweets). Permite evaluar si la información relacional del grafo k-NN beneficia la clasificación cuando los documentos son más largos e informativos por sí mismos.---### 5.4 Comparación entre casos de estudio| Dimensión | Caso 1: Disaster Tweets | Caso 2: AG News ||-----------|------------------------|-----------------|| Tipo de texto | Tweets (~15 palabras) | Noticias (título + descripción) || Clases | 2 (binario) | 4 (multiclase) || Muestras | 6,542 | 24,000 || Features TF-IDF | 3,000 | 5,000 || Nodos en el grafo | 6,542 | 24,000 || Aristas | 32,710 | 120,000 || Desafío principal | Ambigüedad semántica (lenguaje figurado vs literal) | Mayor número de clases y textos más largos || Hipótesis | La estructura de grafo puede ayudar cuando el texto es muy corto y ambiguo | El baseline clásico podría ser competitivo por la riqueza textual |

## 6. Experimentos Computacionales### Configuración experimental por caso| Parámetro | Caso 1 (Disaster Tweets) | Caso 2 (AG News) ||-----------|--------------------------|-------------------|| Muestras totales | 6,542 | 24,000 || Features TF-IDF | 3,000 | 5,000 || k vecinos (k-NN) | 5 | 5 || Número de clases | 2 | 4 || Modelos evaluados | GraphSAGE + GCN | GraphSAGE + GCN || Baseline | Regresión Logística sobre TF-IDF | Regresión Logística sobre TF-IDF || Split train/test | 80/20 estratificado | 80/20 estratificado || Capas GNN | 2 | 2 || Dimensión oculta | 64 | 64 || Dropout | 0.5 | 0.5 || Optimizador | Adam (lr=0.01, weight_decay=5e-4) | Adam (lr=0.01, weight_decay=5e-4) || Early stopping | paciencia=30, máx 200 épocas | paciencia=30, máx 200 épocas || Métricas | Accuracy, Precision, Recall, F1-macro | Accuracy, Precision, Recall, F1-macro |### Justificación de la configuración- **TF-IDF con 3,000/5,000 features:** Suficiente para capturar el vocabulario relevante sin generar grafos excesivamente grandes.- **k=5 vecinos:** Valor estándar en k-NN que produce grafos conexos sin densidad excesiva.- **2 capas GNN:** Permite propagar información a 2 saltos de distancia; más capas pueden causar over-smoothing.- **Dropout 0.5:** Regularización para prevenir overfitting en grafos pequeños.- **Early stopping (paciencia=30):** Evita sobre-entrenamiento y ahorra tiempo computacional.

## 7. Análisis de Resultados> **Pendiente:** Completar tras ejecutar el código de la Parte II. Incrustar aquí las figuras generadas (training curves, matrices de confusión, t-SNE, gráfico de barras).### 7.1 Caso 1 — Disaster Tweets#### Tabla de métricas*(Tabla con Baseline vs GraphSAGE vs GCN para Accuracy, Precision, Recall, F1.)*#### Matriz de confusión*(Imagen de la matriz de confusión.)*#### Curvas de entrenamiento*(Imagen de curvas de pérdida y accuracy.)*#### Proyección t-SNE*(Imagen de visualización t-SNE de embeddings.)*#### Interpretación*(¿El grafo ayudó sobre el baseline? ¿Qué modelo fue mejor? ¿Por qué?)*---### 7.2 Caso 2 — AG News#### Tabla de métricas*(Tabla con Baseline vs GraphSAGE vs GCN para Accuracy, Precision, Recall, F1.)*#### Matriz de confusión*(Imagen de la matriz de confusión.)*#### Curvas de entrenamiento*(Imagen de curvas de pérdida y accuracy.)*#### Proyección t-SNE*(Imagen de visualización t-SNE de embeddings.)*#### Interpretación*(¿El grafo ayudó sobre el baseline? ¿Qué modelo fue mejor? ¿Por qué?)*---### 7.3 Comparativa global#### Gráfico de barras comparativo*(Imagen del gráfico de barras Baseline vs GraphSAGE vs GCN para ambos casos.)*#### Discusión*(¿Cuándo conviene usar GNN sobre métodos clásicos en NLP? ¿En qué tipo de problemas la estructura de grafo aporta valor?)*

## 8. Dificultades encontradas y soluciones### Dificultad 1: Inferencia sobre nuevos textos con GNNs entrenadas sobre un grafo fijo**Problema:** Las GNNs tradicionales (como GCN) operan sobre un grafo fijo. Incorporar un nuevo texto requiere reconstruir el grafo completo o diseñar un mecanismo de inferencia local.**Solución:** Se implementó inferencia por subgrafo local usando `k_hop_subgraph` de PyTorch Geometric: el nuevo texto se conecta por k-NN a sus vecinos más similares, se extrae el subgrafo de k saltos alrededor del nuevo nodo, y se ejecuta el forward pass solo sobre ese subgrafo.### Dificultad 2: Balanceo del dataset AG News**Problema:** El dataset original tiene 120,000 noticias. Entrenar una GNN sobre un grafo de ese tamaño es costoso y puede exceder los recursos disponibles.**Solución:** Se muestrearon 6,000 noticias por clase (24,000 totales), manteniendo el balance entre categorías y logrando un grafo manejable sin sacrificar representatividad estadística.

## 9. Conclusiones> **Pendiente:** Redactar tras completar los experimentos y el análisis de resultados.*(Qué se logró, qué se aprendió sobre GNNs aplicadas a NLP, limitaciones encontradas, posibles extensiones o trabajo futuro.)*

## 10. Bibliografía1. Hamilton, W., Ying, Z., & Leskovec, J. (2017). *Inductive Representation Learning on Large Graphs.* Advances in Neural Information Processing Systems (NeurIPS).2. Veličković, P., Cucurull, G., Casanova, A., Romero, A., Liò, P., & Bengio, Y. (2018). *Graph Attention Networks.* International Conference on Learning Representations (ICLR).3. Kipf, T. N., & Welling, M. (2017). *Semi-Supervised Classification with Graph Convolutional Networks.* International Conference on Learning Representations (ICLR).4. Fey, M., & Lenssen, J. E. (2019). *Fast Graph Representation Learning with PyTorch Geometric.* ICLR Workshop on Representation Learning on Graphs and Manifolds.5. Kaggle. *Natural Language Processing with Disaster Tweets.* https://www.kaggle.com/c/nlp-getting-started6. Zhang, X., Zhao, J., & LeCun, Y. (2015). *Character-level Convolutional Networks for Text Classification.* (AG News dataset). NeurIPS.---## Fin de la Parte I

# PARTE II: CÓDIGO FUENTE DOCUMENTADO**Anexo: Código Fuente Documentado** — A continuación se presenta la implementación completa del software **SentinelGrafo**, desarrollado como parte del Trabajo Computacional Nº 1 del curso Inteligencia Artificial 2026-1, Sección 3.*(El código completo de las 17 secciones se integrará aquí.)*---